# Solar filament segmentation: train to submission

This notebook audits MAGFiLO, trains leakage-safe fold 0, predicts all test images, and decode-validates the final COCO RLE CSV. Enable a Kaggle GPU and attach both the competition data and this repository.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

required = {'cv2': 'opencv-python-headless==4.12.0.88', 'pycocotools': 'pycocotools==2.0.10'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

project_candidates = [Path.cwd(), *Path('/kaggle/input').glob('**/*')] if Path('/kaggle/input').exists() else [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in project_candidates if path.is_dir() and (path / 'solar_filament').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

data_candidates = [
    PROJECT_ROOT / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026',
    Path('/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'),
]
DATA_ROOT = next(path for path in data_candidates if path.exists())
OUTPUT_ROOT = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'artifacts'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_ROOT)
print('data:', DATA_ROOT)
print('output:', OUTPUT_ROOT)

## Audit the attached snapshot

A clean run reports 707 train files, 180 test files, and no errors.

In [ ]:
from solar_filament.data import audit_dataset

audit = audit_dataset(DATA_ROOT)
audit.as_dict()

## Train fold 0

The first run may download ImageNet ResNet50 weights. Set `pretrained_backbone=False` when internet is disabled. For a fast pipeline check, change `epochs` to 1; restore 20 before producing the final candidate.

In [ ]:
from solar_filament.training import TrainConfig, train

config = TrainConfig(
    data_root=str(DATA_ROOT),
    output_dir=str(OUTPUT_ROOT / 'fold-0'),
    fold=0,
    epochs=20,
    image_size=768,
    batch_size=2,
    pretrained_backbone=True,
)
checkpoint = train(config)
checkpoint

## Infer and validate every RLE row

The run manifest records coverage, component areas, and latency for all 180 test images.

In [ ]:
from solar_filament.inference import infer_directory

submission_path = OUTPUT_ROOT / 'submission.csv'
report = infer_directory(
    checkpoint,
    DATA_ROOT / 'test' / 'test_images',
    submission_path,
)
report

In [ ]:
import json

run = json.loads(submission_path.with_suffix('.run.json').read_text())
assert run['processed_images'] == 180
assert not report.errors
print(f"ready: {submission_path} ({report.rows} predicted instances)")